# 02 — SQL Layer
Student Success Analytics Platform

In [1]:
# imports + make src/ importable from inside notebooks/
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.config import PROCESSED_DATA_PATH, DB_PATH, TABLE_NAME
from src.database import load_dataframe, run_query, table_schema, row_count

print("libraries loaded")

libraries loaded


In [2]:
# load cleaned dataset exported at the end of 01_eda.ipynb
df = pd.read_csv(PROCESSED_DATA_PATH)
print("shape:", df.shape)
print("columns:", df.columns.tolist())

shape: (80000, 35)
columns: ['student_id', 'age', 'gender', 'major', 'study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 'attendance_percentage', 'sleep_hours', 'diet_quality', 'exercise_frequency', 'parental_education_level', 'internet_quality', 'mental_health_rating', 'extracurricular_participation', 'previous_gpa', 'semester', 'stress_level', 'dropout_risk', 'social_activity', 'screen_time', 'study_environment', 'access_to_tutoring', 'family_income_range', 'parental_support_level', 'motivation_level', 'exam_anxiety_score', 'learning_style', 'time_management_score', 'exam_score', 'grade_tier', 'wellness_score', 'distraction_hours', 'study_efficiency']


In [3]:
# write the cleaned dataframe into SQLite
load_dataframe(df, table_name=TABLE_NAME, if_exists="replace")
print(f"loaded {row_count()} rows into '{TABLE_NAME}' table at {DB_PATH}")

loaded 80000 rows into 'students' table at C:\Users\lanre\Documents\student-performance-analysis\data\processed\students.db


In [4]:
# confirm schema matches what we expect
schema = table_schema()
print(schema[["name", "type"]].to_string(index=False))

                         name    type
                   student_id INTEGER
                          age INTEGER
                       gender    TEXT
                        major    TEXT
          study_hours_per_day    REAL
           social_media_hours    REAL
                netflix_hours    REAL
                part_time_job    TEXT
        attendance_percentage    REAL
                  sleep_hours    REAL
                 diet_quality    TEXT
           exercise_frequency INTEGER
     parental_education_level    TEXT
             internet_quality    TEXT
         mental_health_rating    REAL
extracurricular_participation    TEXT
                 previous_gpa    REAL
                     semester INTEGER
                 stress_level    REAL
                 dropout_risk    TEXT
              social_activity INTEGER
                  screen_time    REAL
            study_environment    TEXT
           access_to_tutoring    TEXT
          family_income_range    TEXT
       paren

## Query 1 — average exam_score by major

In [5]:
q1 = run_query('''
    SELECT major,
           COUNT(*) AS n_students,
           ROUND(AVG(exam_score), 2) AS avg_exam_score,
           ROUND(AVG(attendance_percentage), 2) AS avg_attendance
    FROM students
    GROUP BY major
    ORDER BY avg_exam_score DESC;
''')
q1

,major,n_students,avg_exam_score,avg_attendance
0,Computer Science,13352,90.27,70.18
1,Engineering,13229,89.10,69.81
2,Business,13276,88.96,69.88
3,Biology,13201,88.85,69.90
4,Psychology,13437,88.84,69.98
5,Arts,13505,88.83,70.06


## Query 2 — dropout rate by attendance bucket
Demonstrates `CASE WHEN` bucketing directly in SQL rather than pandas `.cut()`.

In [6]:
q2 = run_query('''
    SELECT
        CASE
            WHEN attendance_percentage < 60 THEN '<60%'
            WHEN attendance_percentage < 75 THEN '60-75%'
            WHEN attendance_percentage < 90 THEN '75-90%'
            ELSE '90%+'
        END AS attendance_bucket,
        COUNT(*) AS n_students,
        SUM(CASE WHEN dropout_risk = 'Yes' THEN 1 ELSE 0 END) AS dropout_count,
        ROUND(100.0 * SUM(CASE WHEN dropout_risk = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS dropout_rate_pct
    FROM students
    GROUP BY attendance_bucket
    ORDER BY MIN(attendance_percentage);
''')
q2

,attendance_bucket,n_students,dropout_count,dropout_rate_pct
0,<60%,26539,546,2.06
1,60-75%,20204,430,2.13
2,75-90%,19816,359,1.81
3,90%+,13441,247,1.84


## Query 3 — top 15 at-risk students
Low attendance + low previous GPA, ordered worst first. This is the kind of query the future FastAPI `/students/at-risk` endpoint would run.

In [7]:
q3 = run_query('''
    SELECT student_id, attendance_percentage, previous_gpa, study_hours_per_day, dropout_risk, exam_score
    FROM students
    WHERE attendance_percentage < 70 AND previous_gpa < 2.5
    ORDER BY attendance_percentage ASC, previous_gpa ASC
    LIMIT 15;
''')
q3

,student_id,attendance_percentage,previous_gpa,study_hours_per_day,dropout_risk,exam_score
0,107687,40.0,2.42,0.6,No,60
1,119241,40.0,2.44,3.2,Yes,56
2,158142,40.1,2.28,1.4,No,57
3,133223,40.1,2.30,2.2,No,54
4,138809,40.2,2.42,4.9,No,67
5,121835,40.3,2.16,1.7,No,53
6,106555,40.3,2.27,4.0,No,61
7,173749,40.3,2.37,6.0,No,64
8,115949,40.4,2.42,0.0,No,55
9,112297,40.4,2.43,0.6,No,66


In [8]:
print("=" * 50)
print("  SQL LAYER SUMMARY")
print("=" * 50)
print(f"  table: {TABLE_NAME}")
print(f"  rows loaded: {row_count():,}")
print(f"  db file: {DB_PATH}")
print("=" * 50)
print()
print("next: 03_preprocessing.ipynb")

  SQL LAYER SUMMARY
  table: students
  rows loaded: 80,000
  db file: C:\Users\lanre\Documents\student-performance-analysis\data\processed\students.db

next: 03_preprocessing.ipynb
